In [183]:
from torchvision.models import (
    densenet121, resnet50, convnext_small, swin_t, vit_b_16,
    efficientnet_v2_s, mobilenet_v3_large
)
from torchvision import datasets, transforms
import torch
from PIL import Image
from pathlib import Path
from torch.utils.data import DataLoader
import os
from tqdm import tqdm


from ultralytics import YOLO
from PIL import Image, ImageDraw
import base64
import io

In [184]:
VAL_DIR = Path(r"D:\Dataset\Cassava_Leaf_Datasets\Classification\test")

BATCH_SIZE = 32

In [185]:
weights_dir = Path(r"../weights")
yolo_model = YOLO(str(weights_dir / "Yolov8.pt"))  # trained weights

In [186]:
def CropImage(image: Image.Image, box: list):

    x1, y1, x2, y2 = map(int, box)

    w, h = image.size
    x1 = max(0, min(x1, w))
    x2 = max(0, min(x2, w))
    y1 = max(0, min(y1, h))
    y2 = max(0, min(y2, h))

    cropped = image.crop((x1, y1, x2, y2))
    return cropped

In [187]:
def Detect(image: Image.Image):

    #Get start time
    results = yolo_model(image, conf=0.60)
    result = results[0]
    boxes = result.boxes    
    names = result.names

    if boxes is None or len(boxes) == 0:
        return True
        # return {"error": "No detections"}

    best_leaf_box = None
    best_conf = -1.0

    for box in boxes:
        cls = int(box.cls[0])
        label = names[cls]
        conf = float(box.conf[0])

        if label.lower() == "cassava_leaf" and conf > best_conf:
            best_conf = conf
            best_leaf_box = box

    if best_leaf_box is None:
        return True
        # return {"error": "Not Leaf"}

    x1, y1, x2, y2 = map(int, best_leaf_box.xyxy[0].tolist())

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(image.width, x2)
    y2 = min(image.height, y2)

    bbox = [x1, y1, x2, y2]

    cropped = CropImage(image, bbox)
    if cropped is None:
        return True
        # return {"error": "Invalid crop"}

    return cropped


In [188]:
from torch.utils.data import Dataset

class CassavaDetectDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.base_dataset = datasets.ImageFolder(root_dir)  # no transform here
        self.transform = transform
        self.classes = self.base_dataset.classes
        self.skipped = 0

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, label = self.base_dataset[idx]  # img is PIL here (no transform yet)

        # Run YOLO detection & crop
        result = Detect(img)

        if result is True:
            # Detection failed — return original image as fallback
            # (or you can filter these out with a custom collate_fn)
            cropped = img
        else:
            cropped = result  # cropped PIL image

        if self.transform:
            cropped = self.transform(cropped)

        return cropped, label

In [189]:
# Standard ImageNet normalization — compatible with all 7 backbones
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [190]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get the weights directory path relative to this file
weights_dir = Path('../Weights_2')

models_config = {
    "ResNet50":       {"model": resnet50,            "weights": weights_dir / "ResNet50.pth"},
    "ConvNeXt":       {"model": convnext_small,      "weights": weights_dir / "ConvNeXt-S.pth"},
    "SwinT":          {"model": swin_t,              "weights": weights_dir / "Swin-T.pth"},
    "EfficientNetV2": {"model": efficientnet_v2_s,   "weights": weights_dir / "EfficientNet-v2.pth"},
    "Vit-B":          {"model": vit_b_16,            "weights": weights_dir / "Vit-B.pth"},
    "MobileNetV3":    {"model": mobilenet_v3_large,  "weights": weights_dir / "MobileNet-v3.pth"},
    "DenseNet121":    {"model": densenet121,          "weights": weights_dir / "DenseNet121.pth"},
}

loaded_models = {}
class_names = None

In [191]:
for name, cfg in models_config.items():
   
    if not cfg["weights"].exists():
        raise FileNotFoundError(
            f"[{name}] Weight file not found: {cfg['weights']}"
        )


    checkpoint = torch.load(str(cfg["weights"]), map_location=device, weights_only=False)


    if class_names is None and "class_names" in checkpoint:
        class_names = checkpoint["class_names"]


    if "model_state_dict" in checkpoint:
        state_key = "model_state_dict"
    elif "model_state" in checkpoint:
        state_key = "model_state"
    else:
        raise KeyError(
            f"[{name}] Checkpoint must contain 'model_state_dict' or 'model_state'. "
            f"Found keys: {list(checkpoint.keys())}"
        )

    if class_names is not None:
        num_classes = len(class_names)
    else:
       
        state_dict = checkpoint[state_key]
        output_keys = [
            "fc.weight",               # ResNet50
            "classifier.2.weight",     # ConvNeXt
            "head.weight",             # SwinT
            "heads.head.weight",       # ViT-B
            "classifier.1.weight",     # EfficientNetV2
            "classifier.3.weight",     # MobileNetV3
            "classifier.weight",       # DenseNet121
        ]
        num_classes = None
        for key in output_keys:
            if key in state_dict:
                num_classes = state_dict[key].shape[0]
                break
        if num_classes is None:
            raise ValueError(
                f"[{name}] Could not infer num_classes from checkpoint. "
                f"No known output layer key found."
            )

    
    model = cfg["model"](weights=None)

    
    if name == "ResNet50":
        model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
    elif name == "ConvNeXt":
        model.classifier[2] = torch.nn.Linear(model.classifier[2].in_features, num_classes)
    elif name == "SwinT":
        model.head = torch.nn.Linear(model.head.in_features, num_classes)
    elif name == "Vit-B":
        model.heads.head = torch.nn.Linear(model.heads.head.in_features, num_classes)
    elif name == "EfficientNetV2":
        model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, num_classes)
    elif name == "MobileNetV3":
        model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, num_classes)
    elif name == "DenseNet121":
        model.classifier = torch.nn.Linear(model.classifier.in_features, num_classes)

    
    model.load_state_dict(checkpoint[state_key], strict=True)
    model.to(device)
    model.eval()

    loaded_models[name] = model
    print(f"[✓] Loaded {name} — {num_classes} classes")

[✓] Loaded ResNet50 — 4 classes
[✓] Loaded ConvNeXt — 4 classes
[✓] Loaded SwinT — 4 classes
[✓] Loaded EfficientNetV2 — 4 classes
[✓] Loaded Vit-B — 4 classes
[✓] Loaded MobileNetV3 — 4 classes
[✓] Loaded DenseNet121 — 4 classes


In [192]:
# FIX #4: Guard against class_names being None after loading all checkpoints
if class_names is None:
    raise ValueError(
        "No 'class_names' key found in any checkpoint. "
        "Ensure your .pth files include class_names when saved."
    )

print(f"\n[✓] Classes ({len(class_names)}): {class_names}\n")



[✓] Classes (4): ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']



In [193]:
val_dataset = CassavaDetectDataset(VAL_DIR, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
class_names = val_dataset.classes

In [194]:
val_accuracies = {
    "ResNet50": 0.9375,
    "ConvNeXt": 0.9792,
    "SwinT": 0.9375,
    "Vit-B": 0.9208,
    "EfficientNetV2": 0.8333,
    "MobileNetV3": 0.8375,
    "DenseNet121": 0.9625
}

total_acc = sum(val_accuracies.values())
weights = {
    name: acc / total_acc
    for name, acc in val_accuracies.items()
}


In [195]:
@torch.no_grad()
def evaluate_ensemble(val_loader, loaded_models):
    correct = 0
    total = 0

    for images, labels in tqdm(val_loader, desc="Ensemble Validating"):
        images = images.to(device)
        labels = labels.to(device)

        probs_sum = None

        for name, model in loaded_models.items():
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            probs_sum = probs if probs_sum is None else probs_sum + probs

        avg_probs = probs_sum / len(loaded_models)
        preds = torch.argmax(avg_probs, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total

In [196]:
ensemble_acc = evaluate_ensemble(val_loader, loaded_models)

print("====================================")
print(f"Ensemble Validation Accuracy: {ensemble_acc:.4f}")
print("====================================")

Ensemble Validating:   0%|          | 0/5 [00:00<?, ?it/s]


0: 640x640 1 Cassava_Leaf, 560.5ms
Speed: 31.9ms preprocess, 560.5ms inference, 21.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 390.9ms
Speed: 4.6ms preprocess, 390.9ms inference, 4.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 397.9ms
Speed: 4.6ms preprocess, 397.9ms inference, 4.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 360.4ms
Speed: 6.1ms preprocess, 360.4ms inference, 3.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 Cassava_Leafs, 365.0ms
Speed: 6.1ms preprocess, 365.0ms inference, 3.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 357.3ms
Speed: 5.5ms preprocess, 357.3ms inference, 2.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 355.3ms
Speed: 6.1ms preprocess, 355.3ms inference, 2.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 401.6ms
Speed: 5.8ms preprocess

Ensemble Validating:  20%|██        | 1/5 [00:32<02:11, 32.82s/it]


0: 640x640 1 Cassava_Leaf, 373.5ms
Speed: 5.4ms preprocess, 373.5ms inference, 2.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 373.5ms
Speed: 5.4ms preprocess, 373.5ms inference, 2.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 339.1ms
Speed: 5.1ms preprocess, 339.1ms inference, 2.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 357.8ms
Speed: 5.4ms preprocess, 357.8ms inference, 2.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 335.8ms
Speed: 5.3ms preprocess, 335.8ms inference, 2.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 Cassava_Leafs, 372.6ms
Speed: 6.8ms preprocess, 372.6ms inference, 2.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 340.5ms
Speed: 5.6ms preprocess, 340.5ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 340.0ms
Speed: 5.3ms preprocess, 

Ensemble Validating:  40%|████      | 2/5 [01:11<01:48, 36.14s/it]

...


0: 640x640 1 Cassava_Leaf, 368.2ms
Speed: 5.7ms preprocess, 368.2ms inference, 2.8ms postprocess per image at shape (1, 3, 640, 640)

0: 448x640 1 Cassava_Leaf, 354.3ms
Speed: 3.1ms preprocess, 354.3ms inference, 3.8ms postprocess per image at shape (1, 3, 448, 640)

0: 480x640 2 Cassava_Leafs, 357.5ms
Speed: 67.3ms preprocess, 357.5ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Cassava_Leaf, 277.9ms
Speed: 2.6ms preprocess, 277.9ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 Cassava_Leafs, 278.8ms
Speed: 2.9ms preprocess, 278.8ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Cassava_Leaf, 272.7ms
Speed: 3.3ms preprocess, 272.7ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 Cassava_Leaf, 271.9ms
Speed: 2.6ms preprocess, 271.9ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 273.3ms
Speed: 2.5ms preproces

Ensemble Validating:  80%|████████  | 4/5 [02:11<00:31, 31.58s/it]


0: 640x640 1 Cassava_Leaf, 621.9ms
Speed: 4.9ms preprocess, 621.9ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 523.4ms
Speed: 5.5ms preprocess, 523.4ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 496.6ms
Speed: 6.4ms preprocess, 496.6ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 507.1ms
Speed: 5.3ms preprocess, 507.1ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x480 1 Cassava_Leaf, 442.7ms
Speed: 2.1ms preprocess, 442.7ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 480)

0: 640x640 1 Cassava_Leaf, 478.6ms
Speed: 5.8ms preprocess, 478.6ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 408.6ms
Speed: 6.9ms preprocess, 408.6ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cassava_Leaf, 383.7ms
Speed: 4.5ms preprocess, 3

Ensemble Validating: 100%|██████████| 5/5 [02:34<00:00, 30.93s/it]

Ensemble Validation Accuracy: 0.5935
